In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
from IPython.display import display, Markdown

# Semana 7B — Componentes de Series Temporales (Laboratorio)

**Objetivo:** Descomponer series meteorológicas reales del dataset
ClimaLab e identificar tendencia, estacionalidad y ruido.
Comparar el modelo aditivo vs multiplicativo en `tdb` y `ghi`.

**Ejercicios:**

1. Resampling y visualización multiescala
2. Descomposición manual de `tdb`
3. Descomposición automática de `tdb` y `ghi`
4. Comparación aditivo vs multiplicativo

In [ ]:
df = pd.read_parquet("data/ClimaLab_2023-05-31_2025-06-20.parquet")

In [ ]:
# Resamplear a promedios horarios y diarios
tdb_horaria = df["tdb"].resample("1h").mean().dropna()
tdb_diaria = df["tdb"].resample("1D").mean().dropna()
ghi_diaria = df["ghi"].resample("1D").mean().dropna()

---
## 1. Visualización multiescala

Los datos a 1 minuto (~1.08M registros) son demasiado densos para
graficar.  Resampleamos a promedios horarios y diarios para revelar
los distintos componentes a cada escala temporal.

In [ ]:
fig1, axes1 = plt.subplots(3, 1, figsize=(13, 10))

# Panel a: todo el período (diario)
axes1[0].plot(tdb_diaria.index, tdb_diaria.values,
              color="steelblue", lw=0.5)
axes1[0].set_ylabel("Temperatura (°C)")
axes1[0].set_title("a) Todo el período (~2 años, resolución diaria)")

# Panel b: un mes (horario) — elegir julio 2024
_mes = tdb_horaria["2024-07"]
axes1[1].plot(_mes.index, _mes.values, color="steelblue", lw=0.8)
axes1[1].set_ylabel("Temperatura (°C)")
axes1[1].set_title("b) Julio 2024 (resolución horaria)")

# Panel c: una semana (horario) — primera semana de julio 2024
_semana = tdb_horaria["2024-07-01":"2024-07-07"]
axes1[2].plot(_semana.index, _semana.values,
              color="steelblue", lw=1.2, marker=".", markersize=3)
axes1[2].set_ylabel("Temperatura (°C)")
axes1[2].set_xlabel("Fecha")
axes1[2].set_title("c) 1–7 julio 2024 (resolución horaria)")

plt.tight_layout()
plt.show()

> **Lo que se revela en cada escala:**
> - **Panel a (2 años):** Estacionalidad anual clara — verano caliente,
>   invierno frío. ¿Hay tendencia? Difícil de ver a simple vista con
>   solo 2 años.
> - **Panel b (1 mes):** El ciclo diario domina — picos a las 14–16h,
>   valles a las 5–7h. La envolvente varía día a día (nubosidad, frentes).
> - **Panel c (1 semana):** Cada ciclo diario es visible individualmente.
>   El ruido son las fluctuaciones dentro de un mismo día.

---
## 2. Descomposición manual de `tdb` diaria

Repetimos los pasos de la sesión teórica, ahora con datos reales:

1. Media móvil de 30 días → tendencia
2. Restar tendencia
3. Promediar por día del año → estacionalidad
4. Residuo = observado − tendencia − estacionalidad

In [ ]:
_y = tdb_diaria.copy()

# Paso 1: Tendencia (media móvil centrada de 30 días)
_tendencia = _y.rolling(window=30, center=True).mean()

# Paso 2: Restar tendencia
_detrended = _y - _tendencia

# Paso 3: Estacionalidad — promedio por día del año (1–366)
_doy = _detrended.index.dayofyear
_estac_por_doy = _detrended.groupby(_doy).mean()
_estacionalidad = _doy.map(_estac_por_doy)
_estacionalidad.index = _y.index

# Paso 4: Residuo
_residuo = _y - _tendencia - _estacionalidad

fig2, axes2 = plt.subplots(4, 1, figsize=(13, 11), sharex=True)

axes2[0].plot(_y.index, _y.values, "steelblue", lw=0.5)
axes2[0].set_ylabel("°C")
axes2[0].set_title("Observada")

axes2[1].plot(_tendencia.index, _tendencia.values, "crimson", lw=2)
axes2[1].set_ylabel("°C")
axes2[1].set_title("Tendencia (media móvil 30 días)")

axes2[2].plot(_estacionalidad.index, _estacionalidad.values,
              "seagreen", lw=0.8)
axes2[2].set_ylabel("°C")
axes2[2].set_title("Estacionalidad (promedio por día del año)")

axes2[3].plot(_residuo.index, _residuo.values, "gray", lw=0.5)
axes2[3].set_ylabel("°C")
axes2[3].set_xlabel("Fecha")
axes2[3].set_title(
    f"Residuo (σ = {np.nanstd(_residuo.values):.2f} °C)"
)

plt.suptitle(
    "Descomposición manual de tdb (diaria)", fontsize=13, y=1.01
)
plt.tight_layout()
plt.show()

> La descomposición manual captura lo esencial:
> - La **tendencia** sigue el ciclo anual suavizado.
> - La **estacionalidad** es el patrón promedio por día del año.
> - El **residuo** tiene σ ≈ 2–3 °C y debería ser aleatorio.
>
> Ahora comparemos con `seasonal_decompose` automático.

---
## 3. Descomposición automática con statsmodels

### 3.1 `tdb` diaria — modelo aditivo (period = 365)

In [ ]:
_result_tdb = seasonal_decompose(
    tdb_diaria, model="additive", period=365,
)

fig3a = _result_tdb.plot()
fig3a.set_size_inches(13, 10)
fig3a.suptitle(
    "seasonal_decompose — tdb diaria, modelo aditivo, period=365",
    fontsize=13, y=1.01,
)
plt.tight_layout()
plt.show()

### 3.2 `ghi` diaria — modelo multiplicativo (period = 365)

La irradiancia solar tiene amplitud proporcional al nivel: en verano
hay más radiación *y* más variabilidad. El modelo multiplicativo
debería capturar esto mejor.

In [ ]:
# ghi tiene valores cercanos a 0 o negativos (noche) — filtrar solo > 0
# para que el modelo multiplicativo funcione (no puede tener ceros)
_ghi_positiva = ghi_diaria[ghi_diaria > 1].copy()

_result_ghi = seasonal_decompose(
    _ghi_positiva, model="multiplicative", period=365,
)

fig3b = _result_ghi.plot()
fig3b.set_size_inches(13, 10)
fig3b.suptitle(
    "seasonal_decompose — ghi diaria, modelo multiplicativo, period=365",
    fontsize=13, y=1.01,
)
plt.tight_layout()
plt.show()

---
## 4. ¿Aditivo o multiplicativo? — Diagnóstico con datos reales

### 4.1 Diagnóstico DE vs Media

Dividimos cada serie en ventanas de 60 días, calculamos media y
desviación estándar por ventana, y verificamos si hay relación.

In [ ]:
_ventana = 60

fig4a, (ax4a1, ax4a2) = plt.subplots(1, 2, figsize=(12, 5))

for ax, serie, nombre, color in [
    (ax4a1, tdb_diaria, "tdb (°C)", "steelblue"),
    (ax4a2, ghi_diaria.dropna(), "ghi (W/m²)", "darkorange"),
]:
    _medias = []
    _des = []
    _vals = serie.values
    for start in range(0, len(_vals) - _ventana, _ventana):
        _bloque = _vals[start : start + _ventana]
        _bloque = _bloque[~np.isnan(_bloque)]
        if len(_bloque) > 5:
            _medias.append(np.mean(_bloque))
            _des.append(np.std(_bloque))

    ax.scatter(_medias, _des, c=color, s=40, alpha=0.7, edgecolors="white")
    _coef = np.polyfit(_medias, _des, 1)
    _x_fit = np.array([min(_medias), max(_medias)])
    ax.plot(_x_fit, np.polyval(_coef, _x_fit), "k--", lw=1.5,
            label=f"pendiente = {_coef[0]:.4f}")
    ax.set_xlabel("Media de la ventana")
    ax.set_ylabel("DE de la ventana")
    ax.set_title(nombre)
    ax.legend()

plt.suptitle(
    f"Diagnóstico aditivo/multiplicativo (ventanas de {_ventana} días)",
    fontsize=13, y=1.02,
)
plt.tight_layout()
plt.show()

> **Resultado esperado:**
> - **`tdb`:** Pendiente ≈ 0 → la variabilidad no depende del nivel → **aditivo**
> - **`ghi`:** Pendiente > 0 → más irradiancia = más variabilidad → **multiplicativo**

### 4.2 Comparación de residuos

Para confirmar, aplicamos *ambos* modelos a cada variable y comparamos
la varianza de los residuos.  El modelo correcto produce residuos
más homogéneos.

In [ ]:
_ghi_pos = ghi_diaria[ghi_diaria > 1].copy()

_resultados = {}
for nombre, serie in [("tdb", tdb_diaria), ("ghi", _ghi_pos)]:
    for modelo in ["additive", "multiplicative"]:
        _r = seasonal_decompose(serie, model=modelo, period=365)
        _resid = _r.resid.dropna()
        _var = np.var(_resid)
        _resultados[(nombre, modelo)] = _var

_rows = ""
for (var, mod), var_resid in _resultados.items():
    _mark = " ✓" if (
        (var == "tdb" and mod == "additive") or
        (var == "ghi" and mod == "multiplicative")
    ) else ""
    _rows += f"| `{var}` | {mod} | {var_resid:.4f} |{_mark}\n"
display(Markdown(f"""
| Variable | Modelo | Varianza de residuos | Mejor |
|:---|:---|---:|:---|
{_rows}

> El modelo con **menor varianza de residuos** es el que mejor
> captura la estructura de la serie.  Para `tdb` debería ser
> aditivo; para `ghi`, multiplicativo.
"""))

---
## Resumen de la sesión

| Ejercicio | Hallazgo principal |
|:---|:---|
| **Multiescala** | A escala anual se ve el ciclo estacional; a escala semanal se ve el ciclo diario + ruido |
| **Descomposición manual** | Media móvil + promedio por día del año captura lo esencial |
| **`seasonal_decompose`** | Automatiza la descomposición con un período dado |
| **Aditivo vs multiplicativo** | `tdb` → aditivo (variabilidad constante); `ghi` → multiplicativo (variabilidad proporcional al nivel) |

**Próxima semana (8):** Autocorrelación (ACF/PACF), estacionariedad
y control de calidad de la serie temporal.